In [8]:
from transformers import AutoImageProcessor, AutoModelForImageClassification
from PIL import Image
import requests
import torch
import io

model_id = "juppy44/plant-identification-2m-vit-b"

processor = AutoImageProcessor.from_pretrained(model_id)
model = AutoModelForImageClassification.from_pretrained(model_id)

# Updated URL to a publicly available plant image
url = "https://imgs.search.brave.com/6avyySwC5KgrHGm38AKGiRe9guX_rEVlYzjE8dbof1I/rs:fit:500:0:1:0/g:ce/aHR0cHM6Ly93d3cu/cGxhbmV0bmF0dXJh/bC5jb20vd3AtY29u/dGVudC91cGxvYWRz/LzIwMjMvMTAvUHJ1/bmluZy1sZW1vbi10/cmVlLmpwZw"
response = requests.get(url)
response.raise_for_status() # Raise an exception for bad status codes
image = Image.open(io.BytesIO(response.content))

inputs = processor(images=image, return_tensors="pt")
with torch.no_grad():
    logits = model(**inputs).logits

pred = logits.softmax(dim=-1)[0]
topk = torch.topk(pred, k=5)

for prob, idx in zip(topk.values, topk.indices):
    label = model.config.id2label[idx.item()]
    print(f"{label}: {prob.item():.4f}")

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Citrus trifoliata: 0.0975
Murraya paniculata: 0.0501
Terminalia catappa: 0.0175
Didymocheton spectabilis: 0.0127
Duranta erecta: 0.0122


In [11]:
# 5) Exportar para API
# Guardado estándar de Transformers
model.save_pretrained("plant_identification_model")
processor.save_pretrained("plant_identification_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

['plant_identification_model/preprocessor_config.json']

In [12]:
!zip -r plant_identification_model.zip plant_identification_model
from google.colab import files
files.download("plant_identification_model.zip")

  adding: plant_identification_model/ (stored 0%)
  adding: plant_identification_model/config.json (deflated 74%)
  adding: plant_identification_model/preprocessor_config.json (deflated 46%)
  adding: plant_identification_model/model.safetensors (deflated 7%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>